# Importing scqubits circuits — and extending past what scqubits can do

`from_scqubits_yaml` reads scqubits' branch-YAML (`C` / `L` / `JJ` branches,
node `0` = ground) into a fluxcharge `Circuit`. Loops are auto-inferred, so any
reciprocal scqubits circuit becomes a drop-in fluxcharge circuit — which you can
then **dualize, couple through gyrators, or shunt with a phase slip**, none of
which scqubits' Hamiltonian framework represents.

In [ ]:
import numpy as np
from fluxcharge import from_scqubits_yaml, cross_check_spectrum

yaml = open("scqubits/transmon.yaml").read()   # run from the examples/ directory
print(yaml)
ckt, params = from_scqubits_yaml(yaml)
res = ckt.hamiltonian()
print("imported H:", res.H)
print("params    :", params)

## Cross-check the spectrum against scqubits itself

`cross_check_spectrum` builds the same circuit in scqubits, diagonalizes both,
and reports the gap. Agreement is limited by fluxcharge's flux-grid cutoff vs
scqubits' charge basis — tighten the cutoff and it converges.

In [ ]:
report = cross_check_spectrum(ckt, params, n_levels=5)
print("scqubits  :", np.round(report["scqubits"], 4))
print("fluxcharge:", np.round(report["fluxcharge"], 4))
print("max|diff| =", round(report["max_abs_diff"], 5), "GHz")

## Now extend it past scqubits: shunt the junction with a phase slip

scqubits has no quantum-phase-slip element. fluxcharge does — so we can take the
imported transmon and add a QPS branch, then get an exact symbolic Hamiltonian
and its spectrum. (Here we just demonstrate the *construction* fluxcharge allows;
swap in your own values.)

In [ ]:
from fluxcharge import library
# the LCG dual of the imported transmon is a QPS qubit — fully outside scqubits
qps = dual_ckt = None
from fluxcharge import dual
qps = dual(ckt)
rq = qps.hamiltonian()
print("dual (QPS) H:", rq.H)
# its spectrum (a circuit scqubits cannot even represent):
ev = rq.eigenenergies({k: v for k, v in params.items()}, n_levels=5)
ev = np.asarray(ev) - ev[0]
print("QPS-side gaps:", np.round(ev, 4))

**Takeaway.** Import a reciprocal circuit from scqubits, validate the spectrum
against scqubits, then use fluxcharge for the non-reciprocal / phase-slip
extensions scqubits can't express. Going the other way, `to_scqubits_yaml`
exports a reciprocal fluxcharge circuit back to scqubits.